# 07_cnn_seq

对应 `index.md` 第 7 阶段：小 CNN + RNN / 序列基础。

笔记本在 `codes/pytorch/07_cnn_seq/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "07_cnn_seq" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "pytorch" / "07_cnn_seq"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/pytorch/07_cnn_seq')

## 1. 一层 Conv2d

用 `nn.Conv2d` 建卷积层：输入通道 `1`，输出通道 `4`，卷积核 `3×3`，`padding=1`。

对已有的 `x`（形状 `N×C×H×W`）做一次前向，打印输出的 `shape`。


In [2]:
import torch
from torch import nn

x = torch.randn(2, 1, 8, 8)

# 作答

conv = nn.Conv2d(1, 4, 3, padding=1)

print(conv(x).shape)
#评阅
# 对。padding=1 保空间，输出 [2, 4, 8, 8]。

#参考答案
# print(nn.Conv2d(1, 4, 3, padding=1)(x).shape)

/Users/keyficller/Documents/AEFS-Notes/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([2, 4, 8, 8])


## 2. MaxPool 再看形状

已有特征图 `feat`。用 `nn.MaxPool2d(kernel_size=2)` 做一次下采样，打印池化后的 `shape`。


In [3]:
import torch
from torch import nn

feat = torch.randn(2, 4, 8, 8)

# 作答

pool = nn.MaxPool2d(2)
print(pool(feat).shape)
#评阅
# 对。2×2 池化，[2, 4, 8, 8] → [2, 4, 4, 4]。

#参考答案
# print(nn.MaxPool2d(2)(feat).shape)


torch.Size([2, 4, 4, 4])


## 3. 小 CNN 前向

用 `nn.Sequential` 搭一个小 CNN：

`Conv2d(1, 8, 3, padding=1)` → `ReLU` → `MaxPool2d(2)` → `Flatten` → `Linear(8 * 4 * 4, 3)`

（输入是 `1×8×8`，池化后空间是 `4×4`。）

对 `x` 前向，打印输出的 `shape`。


In [6]:
import torch
from torch import nn

x = torch.randn(2, 1, 8, 8)

# 作答

net = nn.Sequential(
    # batch = 2
    nn.Conv2d(1, 8, 3, padding=1), # 1x8x8 -> 8x8x8
    nn.ReLU(),                     # 8x8x8 -> 8x8x8
    nn.MaxPool2d(2),               # 8x8x8 -> 4x4x8
    nn.Flatten(),                  # 4x4x8 -> 128
    nn.Linear(8 * 4 * 4, 3)        # 128 -> 3
)

print(net(x).shape)
#评阅
# 对。结构与输出 [2, 3] 都对。注释里池化后写成 4x4x8，按 NCHW 其实是 8×4×4，不影响代码。

#参考答案
# net = nn.Sequential(
#     nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
#     nn.Flatten(), nn.Linear(8 * 4 * 4, 3),
# )
# print(net(x).shape)

torch.Size([2, 3])


## 4. 一层 RNN

用 `nn.RNN`：`input_size=5`，`hidden_size=7`，`batch_first=True`。

对已有的 `x`（形状 `batch × seq × feature`）前向，打印输出序列 `out` 的 `shape`，以及最终隐状态 `h_n` 的 `shape`。


In [7]:
import torch
from torch import nn

x = torch.randn(3, 4, 5)

# 作答

rnn = nn.RNN(
    input_size=5,
    hidden_size=7,
    batch_first=True
)

out, hidden = rnn(x)
print(out.shape)
print(hidden.shape)
#评阅
# 对。out 为 [batch, seq, hidden]=[3,4,7]；h_n 为 [num_layers, batch, hidden]=[1,3,7]。

#参考答案
# out, h_n = nn.RNN(5, 7, batch_first=True)(x)
# print(out.shape, h_n.shape)


torch.Size([3, 4, 7])
torch.Size([1, 3, 7])


## 5. LSTM 取最后一步

已有 `nn.LSTM`（`batch_first=True`）和输入 `x`。前向得到 `out, (h_n, c_n)`。

从 `out` 取出**每个序列最后一步**的向量（形状应为 `batch × hidden`），打印这个张量的 `shape`。


In [8]:
import torch
from torch import nn

lstm = nn.LSTM(input_size=6, hidden_size=8, batch_first=True)
x = torch.randn(2, 5, 6)

# 作答

out, (h_n, c_n) = lstm(x)
print(out[:, -1, :].shape)
#评阅
# 对。batch_first 下用 out[:, -1, :] 取末步，shape [2, 8]。

#参考答案
# out, _ = lstm(x)
# print(out[:, -1, :].shape)


torch.Size([2, 8])
